<a href="https://www.kaggle.com/code/nmavros/deepfire-forecaster?scriptVersionId=300677095" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [4]:
import os
import glob
import random
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
import rasterio
from rasterio.windows import Window

print("Initializing Data Pipeline...")

KAGGLE_DATA_ROOT = "/kaggle/input/ts-satfire/ts-satfire"
TARGET_SIZE = 256
SEQUENCE_LENGTH = 3
BATCH_SIZE = 8

# --- 1. Event Level Discovery and Splitting ---
all_dirs = sorted([os.path.join(KAGGLE_DATA_ROOT, d) for d in os.listdir(KAGGLE_DATA_ROOT) 
                   if os.path.isdir(os.path.join(KAGGLE_DATA_ROOT, d))])

valid_events = []
for d in all_dirs:
    lulc_files = glob.glob(os.path.join(d, "ESRI_LULC", "*.tif"))
    viirs_files = sorted(glob.glob(os.path.join(d, "VIIRS_Day", "*.tif")))
    target_files = sorted(glob.glob(os.path.join(d, "FirePred", "*.tif")))
    if len(lulc_files) > 0 and len(viirs_files) >= SEQUENCE_LENGTH + 1 and len(target_files) > 0:
        valid_events.append(d)

random.seed(42)
random.shuffle(valid_events)

n_total = len(valid_events)
n_train = int(0.7 * n_total)
n_val = int(0.15 * n_total)

train_events = valid_events[:n_train]
val_events = valid_events[n_train:n_train + n_val]
test_events = valid_events[n_train + n_val:]

print(f"Total Valid Events: {n_total}")

# --- 2. Dataset Class ---
class SatFireDataset(Dataset):
    def __init__(self, event_list, seq_len=SEQUENCE_LENGTH):
        self.seq_len = seq_len
        self.samples = [] 
        for d in event_list:
            viirs_files = sorted(glob.glob(os.path.join(d, "VIIRS_Day", "*.tif")))
            target_files = sorted(glob.glob(os.path.join(d, "FirePred", "*.tif")))
            with rasterio.open(target_files[0]) as src:
                num_target_bands = src.count
            num_possible_windows = min(len(viirs_files) - self.seq_len, num_target_bands - self.seq_len)
            for t in range(num_possible_windows):
                self.samples.append((d, t, viirs_files, target_files[0]))

    def __len__(self):
        return len(self.samples)
    
    def _normalize(self, tensor):
        min_val = tensor.min()
        max_val = tensor.max()
        if max_val - min_val > 1e-6:
            return (tensor - min_val) / (max_val - min_val)
        return tensor

    def _load_and_interpolate(self, filepath, is_categorical=False, is_massive=False, band_idx=None):
        with rasterio.open(filepath) as src:
            if is_massive:
                window = Window(col_off=0, row_off=0, width=TARGET_SIZE, height=TARGET_SIZE)
                image_data = src.read(window=window)
            else:
                if band_idx is not None:
                    image_data = src.read(band_idx)
                    image_data = image_data.reshape(1, image_data.shape[0], image_data.shape[1])
                else:
                    image_data = src.read()
                
            tensor = torch.tensor(image_data, dtype=torch.float32).unsqueeze(0)
            
            if is_categorical:
                tensor = F.interpolate(tensor, size=(TARGET_SIZE, TARGET_SIZE), mode='nearest')
            else:
                tensor = F.interpolate(tensor, size=(TARGET_SIZE, TARGET_SIZE), mode='bilinear', align_corners=False)
                
            tensor = tensor.squeeze(0)
            if not is_categorical:
                tensor = self._normalize(tensor)
            return tensor

    def __getitem__(self, idx):
        event_path, start_t, viirs_list, target_file = self.samples[idx]
        
        lulc_file = glob.glob(os.path.join(event_path, "ESRI_LULC", "*.tif"))[0]
        lulc_tensor = self._load_and_interpolate(lulc_file, is_categorical=True, is_massive=True)
        
        viirs_sequence = []
        for t_offset in range(self.seq_len):
            day_tensor = self._load_and_interpolate(viirs_list[start_t + t_offset], is_categorical=False)
            viirs_sequence.append(day_tensor)
        viirs_tensor = torch.stack(viirs_sequence, dim=0)
        
        target_band = start_t + self.seq_len + 1
        target_tensor = self._load_and_interpolate(target_file, is_categorical=False, band_idx=target_band)
        
        return viirs_tensor, lulc_tensor, target_tensor

# --- 3. Instantiate DataLoaders ---
train_loader = DataLoader(SatFireDataset(train_events), batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(SatFireDataset(val_events), batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(SatFireDataset(test_events), batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print("DataLoaders built successfully.")

Initializing Data Pipeline...
Total Valid Events: 149
DataLoaders built successfully.


In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F

print("Initializing Model Architecture and Loss Function...")

# --- 1. SPATIAL ENCODER ---
class CNNBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.conv(x)

class SpatialEncoder(nn.Module):
    def __init__(self, viirs_channels=8, lulc_channels=1, embed_dim=128):
        super().__init__()
        in_channels = viirs_channels + lulc_channels
        self.layer1 = CNNBlock(in_channels, 32)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.layer2 = CNNBlock(32, 64)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.layer3 = CNNBlock(64, embed_dim)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(self, viirs_seq, lulc):
        B, T, C_v, H, W = viirs_seq.shape
        C_l = lulc.shape[1]
        
        viirs_seq = torch.nan_to_num(viirs_seq, nan=0.0, posinf=0.0, neginf=0.0)
        lulc = torch.nan_to_num(lulc, nan=0.0, posinf=0.0, neginf=0.0)
        
        viirs_reshaped = viirs_seq.view(B * T, C_v, H, W)
        lulc_repeated = lulc.unsqueeze(1).repeat(1, T, 1, 1, 1)
        lulc_reshaped = lulc_repeated.view(B * T, C_l, H, W)
        
        x = torch.cat([viirs_reshaped, lulc_reshaped], dim=1)
        
        x = self.pool1(self.layer1(x))
        x = self.pool2(self.layer2(x))
        features = self.pool3(self.layer3(x))
        
        _, C_f, H_f, W_f = features.shape
        return features.view(B, T, C_f, H_f, W_f)

# --- 2. SPATIOTEMPORAL TRANSFORMER ---
class SimpleSpatiotemporalTransformer(nn.Module):
    def __init__(self, embed_dim=128, num_heads=4, num_layers=2, seq_len=3, spatial_size=32):
        super().__init__()
        self.num_tokens = seq_len * spatial_size * spatial_size
        self.pos_embed = nn.Parameter(torch.randn(1, self.num_tokens, embed_dim) * 0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads, dim_feedforward=embed_dim * 2,
            dropout=0.1, activation='gelu', batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

    def forward(self, features_temporal):
        B, T, C, H, W = features_temporal.shape
        x = features_temporal.view(B, T * H * W, C)
        x = x + self.pos_embed
        x = self.transformer(x)
        return x.view(B, T, C, H, W)

# --- 3. ATTENTION POOLING & DECODER ---
class TemporalAttentionPooling(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.attention_net = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // 2, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels // 2, 1, kernel_size=1)
        )

    def forward(self, x):
        B, T, C, H, W = x.shape
        x_reshaped = x.view(B * T, C, H, W)
        scores = self.attention_net(x_reshaped).view(B, T, -1).mean(dim=2)
        weights = F.softmax(scores, dim=1)
        weights_expanded = weights.view(B, T, 1, 1, 1)
        pooled_features = torch.sum(x * weights_expanded, dim=1)
        return pooled_features, weights

class GenerativeDecoder(nn.Module):
    def __init__(self, in_channels=128):
        super().__init__()
        self.up1 = nn.Sequential(nn.ConvTranspose2d(in_channels, 64, kernel_size=4, stride=2, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True))
        self.up2 = nn.Sequential(nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True))
        self.up3 = nn.Sequential(nn.ConvTranspose2d(32, 16, kernel_size=4, stride=2, padding=1), nn.BatchNorm2d(16), nn.ReLU(inplace=True))
        self.final_conv = nn.Conv2d(16, 1, kernel_size=3, padding=1)
        
    def forward(self, x):
        return self.final_conv(self.up3(self.up2(self.up1(x))))

# --- 4. FULL MODEL ASSEMBLY ---
class SatGenTransformer(nn.Module):
    def __init__(self, embed_dim=128):
        super().__init__()
        self.spatial_encoder = SpatialEncoder(embed_dim=embed_dim)
        self.transformer = SimpleSpatiotemporalTransformer(embed_dim=embed_dim)
        self.attention_pool = TemporalAttentionPooling(in_channels=embed_dim)
        self.decoder = GenerativeDecoder(in_channels=embed_dim)

    def forward(self, viirs_seq, lulc):
        features = self.spatial_encoder(viirs_seq, lulc)
        attended_features = self.transformer(features)
        pooled_features, attn_weights = self.attention_pool(attended_features)
        logits = self.decoder(pooled_features)
        return logits, attn_weights

# --- 5. CUSTOM LOSS FUNCTION ---
class MaskedFocalDiceLoss(nn.Module):
    def __init__(self, alpha=0.8, gamma=2.0, smooth=1e-6):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.smooth = smooth

    def forward(self, logits, targets):
        valid_mask = ~torch.isnan(targets)
        
        # Safe Binarization to prevent GPU crashes
        targets_clean = torch.nan_to_num(targets, nan=0.0)
        targets_clean = (targets_clean > 0).float()
        targets_clean = torch.clamp(targets_clean, min=0.0, max=1.0)
        
        probs = torch.sigmoid(logits)
        
        probs_flat = probs.view(-1) * valid_mask.view(-1).float()
        targets_flat = targets_clean.view(-1) * valid_mask.view(-1).float()
        mask_sum = valid_mask.view(-1).float().sum() + self.smooth
        
        bce = F.binary_cross_entropy(probs_flat, targets_flat, reduction='none')
        focal_loss = (self.alpha * (1 - torch.exp(-bce)) ** self.gamma * bce).sum() / mask_sum
        
        intersection = (probs_flat * targets_flat).sum()
        dice_loss = 1 - (2. * intersection + self.smooth) / (probs_flat.sum() + targets_flat.sum() + self.smooth)
        
        return focal_loss + dice_loss

# --- GPU DRY RUN TEST ---
print("Running safety check on GPU...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

test_model = SatGenTransformer().to(device)
test_criterion = MaskedFocalDiceLoss()

dummy_v = torch.randn(2, 3, 8, 256, 256).to(device)
dummy_l = torch.randn(2, 1, 256, 256).to(device)
# Simulating a harsh target with negative values and NaNs to test the clamp
dummy_t = torch.randn(2, 1, 256, 256).to(device) 
dummy_t[0, 0, 0, 0] = float('nan')
dummy_t[0, 0, 1, 1] = -9999.0

out, _ = test_model(dummy_v, dummy_l)
loss = test_criterion(out, dummy_t)

print(f"Safety check passed! Simulated Loss: {loss.item():.4f}")
print("Architecture loaded successfully.")

Initializing Model Architecture and Loss Function...
Running safety check on GPU...
Safety check passed! Simulated Loss: 0.7100
Architecture loaded successfully.


In [ ]:
import torch
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

print("============================================================")
print("Initializing Training Engine & Evaluation Pipeline")
print("============================================================\n")

# --- 1. SETUP ---
EPOCHS = 10
EMBED_DIM = 128
SAVE_PATH = "novhyt_best_model.pth"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SatGenTransformer(embed_dim=EMBED_DIM).to(device)
criterion = MaskedFocalDiceLoss(alpha=0.8, gamma=2.0)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

train_losses = []
val_losses = []
best_val_loss = float('inf')

print(f"Starting training for {EPOCHS} epochs on {device}...\n")

# --- 2. TRAINING LOOP ---
for epoch in range(EPOCHS):
    # Training Phase
    model.train()
    epoch_train_loss = 0.0
    train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")
    
    for viirs_batch, lulc_batch, target_batch in train_pbar:
        viirs_batch = viirs_batch.to(device)
        lulc_batch = lulc_batch.to(device)
        target_batch = target_batch.to(device)
        
        optimizer.zero_grad()
        logits, _ = model(viirs_batch, lulc_batch)
        loss = criterion(logits, target_batch)
        loss.backward()
        optimizer.step()
        
        epoch_train_loss += loss.item()
        train_pbar.set_postfix({'loss': f"{loss.item():.4f}"})
        
    avg_train_loss = epoch_train_loss / len(train_loader)
    train_losses.append(avg_train_loss)
    
    # Validation Phase
    model.eval()
    epoch_val_loss = 0.0
    with torch.no_grad():
        val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Valid]")
        for viirs_batch, lulc_batch, target_batch in val_pbar:
            viirs_batch = viirs_batch.to(device)
            lulc_batch = lulc_batch.to(device)
            target_batch = target_batch.to(device)
            
            logits, _ = model(viirs_batch, lulc_batch)
            loss = criterion(logits, target_batch)
            
            epoch_val_loss += loss.item()
            val_pbar.set_postfix({'loss': f"{loss.item():.4f}"})
            
    avg_val_loss = epoch_val_loss / len(val_loader)
    val_losses.append(avg_val_loss)
    
    print(f"--> Summary | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
    
    # Checkpoint Saving
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), SAVE_PATH)
        print(f"    [*] Validation loss improved! Model saved.")

print("\nTraining Complete! Generating Paper Visualizations...\n")

# --- 3. LEARNING CURVE PLOT ---
plt.figure(figsize=(10, 6))
plt.plot(range(1, EPOCHS+1), train_losses, label='Train Loss', marker='o', color='blue')
plt.plot(range(1, EPOCHS+1), val_losses, label='Validation Loss', marker='s', color='orange')
plt.title('Learning Curve: Masked Focal-Dice Loss per Epoch')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend()
plt.savefig('learning_curve.png', bbox_inches='tight', dpi=300)
plt.show()

# --- 4. INFERENCE & ERROR MAP ---
print("Running inference on unseen test data...")

# Load the best weights
model.load_state_dict(torch.load(SAVE_PATH))
model.eval()

# Grab one batch from the Test Set
viirs_batch, lulc_batch, target_batch = next(iter(test_loader))

with torch.no_grad():
    logits, _ = model(viirs_batch[0:1].to(device), lulc_batch[0:1].to(device))
    pred_prob = torch.sigmoid(logits).cpu()[0, 0]

# Binarize Predictions (Threshold = 0.5) and Targets
pred_bin = (pred_prob > 0.5).int().numpy()
target_clean = torch.nan_to_num(target_batch[0, 0], nan=0.0)
target_bin = (target_clean > 0).int().numpy()

# Build RGB Error Map
error_map = np.ones((256, 256, 3)) * 0.9 # Light Gray Background
error_map[(pred_bin == 1) & (target_bin == 1)] = [0.0, 0.0, 1.0] # True Positive (Blue)
error_map[(pred_bin == 1) & (target_bin == 0)] = [1.0, 0.0, 0.0] # False Positive (Red)
error_map[(pred_bin == 0) & (target_bin == 1)] = [1.0, 0.0, 0.0] # False Negative (Red)

# Plotting the 6 Panels
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Spatiotemporal Fire Prediction: Model Evaluation', fontsize=18)

for i in range(3):
    axes[0, i].imshow(viirs_batch[0, i, 0].cpu().numpy(), cmap='viridis')
    axes[0, i].set_title(f"Input Day {i+1} (Thermal Sequence)")
    axes[0, i].axis('off')

axes[1, 0].imshow(target_clean.numpy(), cmap='inferno')
axes[1, 0].set_title("Actual Fire Truth (Day 4)")
axes[1, 0].axis('off')

axes[1, 1].imshow(pred_prob.numpy(), cmap='inferno')
axes[1, 1].set_title("Model Prediction Probabilities")
axes[1, 1].axis('off')

axes[1, 2].imshow(error_map)
axes[1, 2].set_title("Error Map (Blue=Hit, Red=Miss)")
axes[1, 2].axis('off')

plt.tight_layout()
plt.savefig('evaluation_maps.png', bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

print("============================================================")
print("Starting Comprehensive Test Set Evaluation")
print("============================================================\n")

# Extract the dataset directly from the dataloader
test_dataset = test_loader.dataset

# --- 1. SETUP DIRECTORIES ---
EVAL_DIR = "test_evaluation"
dirs = ["best", "worst", "random"]
for d in dirs:
    os.makedirs(os.path.join(EVAL_DIR, d), exist_ok=True)

# Load the winning model from Epoch 8
model.load_state_dict(torch.load(SAVE_PATH))
model.eval()

# --- 2. EVALUATION LOOP ---
results = []
global_attention = np.zeros(3)

total_tp = 0
total_fp = 0
total_fn = 0

print("Scanning all Test Sequences and computing metrics...")
with torch.no_grad():
    for i in tqdm(range(len(test_dataset))):
        viirs_seq, lulc, target = test_dataset[i]
        
        # Add batch dimension and move to GPU
        v_in = viirs_seq.unsqueeze(0).to(device)
        l_in = lulc.unsqueeze(0).to(device)
        
        logits, attn = model(v_in, l_in)
        pred_prob = torch.sigmoid(logits)[0, 0].cpu()
        
        # Binarize
        pred_bin = (pred_prob > 0.5).int()
        target_clean = torch.nan_to_num(target[0], nan=0.0)
        target_bin = (target_clean > 0).int()
        
        # Calculate Metrics for this sequence
        tp = (pred_bin & target_bin).sum().item()
        fp = (pred_bin & ~target_bin).sum().item()
        fn = (~pred_bin & target_bin).sum().item()
        
        total_tp += tp
        total_fp += fp
        total_fn += fn
        
        # FIX: Flatten the attention array to match the (3,) shape
        global_attention += attn.cpu().numpy().flatten()
        
        # We only rank events that actually had some fire in the target
        if target_bin.sum() > 10: 
            iou = tp / (tp + fp + fn + 1e-6)
            results.append({'idx': i, 'iou': iou, 'tp': tp, 'fp': fp, 'fn': fn})

# --- 3. SORTING AND SELECTING SAMPLES ---
results.sort(key=lambda x: x['iou'], reverse=True)

top_10 = results[:10]
bottom_10 = results[-10:]
random_10 = np.random.choice(results[10:-10], min(10, len(results)-20), replace=False).tolist() if len(results) > 20 else []

selections = {
    'best': top_10,
    'worst': bottom_10,
    'random': random_10
}

# --- 4. VISUALIZATION ENGINE ---
def save_evaluation_plot(idx, category, rank, iou_score):
    viirs_seq, lulc, target = test_dataset[idx]
    v_in = viirs_seq.unsqueeze(0).to(device)
    l_in = lulc.unsqueeze(0).to(device)
    
    with torch.no_grad():
        logits, _ = model(v_in, l_in)
        pred_prob = torch.sigmoid(logits)[0, 0].cpu()

    pred_bin = (pred_prob > 0.5).int().numpy()
    target_clean = torch.nan_to_num(target[0], nan=0.0)
    target_bin = (target_clean > 0).int().numpy()

    error_map = np.ones((256, 256, 3)) * 0.9 
    error_map[(pred_bin == 1) & (target_bin == 1)] = [0.0, 0.0, 1.0] # Hit
    error_map[(pred_bin == 1) & (target_bin == 0)] = [1.0, 0.0, 0.0] # False Alarm
    error_map[(pred_bin == 0) & (target_bin == 1)] = [1.0, 0.0, 0.0] # Missed Fire

    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle(f'Category: {category.upper()} | Rank: {rank} | IoU Score: {iou_score:.4f}', fontsize=16)

    for i in range(3):
        axes[0, i].imshow(viirs_seq[i, 0].numpy(), cmap='viridis')
        axes[0, i].set_title(f"Input Day {i+1} (Thermal)")
        axes[0, i].axis('off')

    axes[1, 0].imshow(target_clean.numpy(), cmap='inferno')
    axes[1, 0].set_title("Actual Fire Truth (Day 4)")
    axes[1, 0].axis('off')

    axes[1, 1].imshow(pred_prob.numpy(), cmap='inferno')
    axes[1, 1].set_title("Model Prediction Probabilities")
    axes[1, 1].axis('off')

    axes[1, 2].imshow(error_map)
    axes[1, 2].set_title("Error Map (Blue=Hit, Red=Miss)")
    axes[1, 2].axis('off')

    plt.tight_layout()
    filename = f"{EVAL_DIR}/{category}/{category}_rank{rank:02d}_idx{idx}.png"
    plt.savefig(filename, bbox_inches='tight', dpi=150)
    plt.close(fig) 

print("\nGenerating Visualizations for Selected Samples...")
for category, samples in selections.items():
    if not samples:
        continue
    print(f"Rendering {category.upper()} maps...")
    for rank, sample in enumerate(tqdm(samples)):
        save_evaluation_plot(sample['idx'], category, rank+1, sample['iou'])

# --- 5. FINAL STATISTICAL REPORT ---
global_precision = total_tp / (total_tp + total_fp + 1e-6)
global_recall = total_tp / (total_tp + total_fn + 1e-6)
global_iou = total_tp / (total_tp + total_fp + total_fn + 1e-6)

attn_percentages = (global_attention / (np.sum(global_attention) + 1e-6)) * 100

print("\n" + "="*50)
print("FINAL TEST SET ACADEMIC REPORT")
print("="*50)
print(f"1. Overall Spatial Accuracy (IoU): {global_iou*100:.2f}%")
print(f"2. Precision (When it predicts fire, is it right?): {global_precision*100:.2f}%")
print(f"3. Recall (How much of the real fire did it catch?): {global_recall*100:.2f}%")
print("-" * 50)
print("SPATIOTEMPORAL ATTENTION ANALYSIS:")
print("How the Transformer distributed its focus across time:")
print(f"- Day 1 (Oldest): {attn_percentages[0]:.1f}% weight")
print(f"- Day 2 (Middle): {attn_percentages[1]:.1f}% weight")
print(f"- Day 3 (Latest): {attn_percentages[2]:.1f}% weight")
print("="*50)
print("\nEvaluation complete! You can now zip and download the 'test_evaluation' folder.")

In [ ]:
!zip -r fast_results.zip /kaggle/working/test_evaluation
!zip fast_results.zip /kaggle/working/novhyt_best_model.pth /kaggle/working/learning_curve.png

from IPython.display import FileLink
display(FileLink('fast_results.zip'))

In [7]:
import os
import glob
import rasterio
import numpy as np

print("Analyzing actual values of the FirePred file...\n")

KAGGLE_DATA_ROOT = "/kaggle/input/ts-satfire/ts-satfire"

# Safely find all valid directories
all_dirs = sorted([os.path.join(KAGGLE_DATA_ROOT, d) for d in os.listdir(KAGGLE_DATA_ROOT) 
                   if os.path.isdir(os.path.join(KAGGLE_DATA_ROOT, d))])

target_file = None
for directory in all_dirs:
    possible_files = glob.glob(os.path.join(directory, "FirePred", "*.tif"))
    if len(possible_files) > 0:
        target_file = possible_files[0]
        break

if target_file is None:
    print("Error: Could not find any FirePred files in the dataset.")
else:
    print(f"Successfully loaded target file: {target_file}")
    with rasterio.open(target_file) as src:
        # Read the 4th band (Day 4 target)
        data = src.read(4) 
        
    # Filter out missing data (NaNs) for accurate statistics
    data_clean = data[~np.isnan(data)]

    print("-" * 50)
    print(f"Minimum value (Min): {data_clean.min()}")
    print(f"Maximum value (Max): {data_clean.max()}")
    print(f"Mean value: {data_clean.mean():.4f}")
    print(f"95th Percentile: {np.percentile(data_clean, 95):.4f}")
    print(f"99th Percentile: {np.percentile(data_clean, 99):.4f}")
    print("-" * 50)

Analyzing actual values of the FirePred file...

Successfully loaded target file: /kaggle/input/ts-satfire/ts-satfire/20562846/FirePred/2017-04-28_FirePred.tif
--------------------------------------------------
Minimum value (Min): 5.300000190734863
Maximum value (Max): 9.699999809265137
Mean value: 7.3753
95th Percentile: 8.8000
99th Percentile: 9.2000
--------------------------------------------------


In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F

print("Initializing Regression-based Architecture and Loss Function...")

class CNNBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.conv(x)

class SpatialEncoder(nn.Module):
    def __init__(self, viirs_channels=8, lulc_channels=1, embed_dim=128):
        super().__init__()
        in_channels = viirs_channels + lulc_channels
        self.layer1 = CNNBlock(in_channels, 32)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.layer2 = CNNBlock(32, 64)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.layer3 = CNNBlock(64, embed_dim)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(self, viirs_seq, lulc):
        B, T, C_v, H, W = viirs_seq.shape
        C_l = lulc.shape[1]
        
        viirs_seq = torch.nan_to_num(viirs_seq, nan=0.0)
        lulc = torch.nan_to_num(lulc, nan=0.0)
        
        viirs_reshaped = viirs_seq.view(B * T, C_v, H, W)
        lulc_repeated = lulc.unsqueeze(1).repeat(1, T, 1, 1, 1)
        lulc_reshaped = lulc_repeated.view(B * T, C_l, H, W)
        
        x = torch.cat([viirs_reshaped, lulc_reshaped], dim=1)
        
        x = self.pool1(self.layer1(x))
        x = self.pool2(self.layer2(x))
        features = self.pool3(self.layer3(x))
        
        _, C_f, H_f, W_f = features.shape
        return features.view(B, T, C_f, H_f, W_f)

class SimpleSpatiotemporalTransformer(nn.Module):
    def __init__(self, embed_dim=128, num_heads=4, num_layers=2, seq_len=3, spatial_size=32):
        super().__init__()
        self.num_tokens = seq_len * spatial_size * spatial_size
        self.pos_embed = nn.Parameter(torch.randn(1, self.num_tokens, embed_dim) * 0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads, dim_feedforward=embed_dim * 2,
            dropout=0.1, activation='gelu', batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

    def forward(self, features_temporal):
        B, T, C, H, W = features_temporal.shape
        x = features_temporal.view(B, T * H * W, C)
        x = x + self.pos_embed
        x = self.transformer(x)
        return x.view(B, T, C, H, W)

class TemporalAttentionPooling(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.attention_net = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // 2, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels // 2, 1, kernel_size=1)
        )

    def forward(self, x):
        B, T, C, H, W = x.shape
        x_reshaped = x.view(B * T, C, H, W)
        scores = self.attention_net(x_reshaped).view(B, T, -1).mean(dim=2)
        weights = F.softmax(scores, dim=1)
        weights_expanded = weights.view(B, T, 1, 1, 1)
        pooled_features = torch.sum(x * weights_expanded, dim=1)
        return pooled_features, weights

class GenerativeDecoder(nn.Module):
    def __init__(self, in_channels=128):
        super().__init__()
        self.up1 = nn.Sequential(nn.ConvTranspose2d(in_channels, 64, kernel_size=4, stride=2, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True))
        self.up2 = nn.Sequential(nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True))
        self.up3 = nn.Sequential(nn.ConvTranspose2d(32, 16, kernel_size=4, stride=2, padding=1), nn.BatchNorm2d(16), nn.ReLU(inplace=True))
        self.final_conv = nn.Conv2d(16, 1, kernel_size=3, padding=1)
        
    def forward(self, x):
        return self.final_conv(self.up3(self.up2(self.up1(x))))

class SatGenTransformer(nn.Module):
    def __init__(self, embed_dim=128):
        super().__init__()
        self.spatial_encoder = SpatialEncoder(embed_dim=embed_dim)
        self.transformer = SimpleSpatiotemporalTransformer(embed_dim=embed_dim)
        self.attention_pool = TemporalAttentionPooling(in_channels=embed_dim)
        self.decoder = GenerativeDecoder(in_channels=embed_dim)

    def forward(self, viirs_seq, lulc):
        features = self.spatial_encoder(viirs_seq, lulc)
        attended_features = self.transformer(features)
        pooled_features, attn_weights = self.attention_pool(attended_features)
        # Raw linear output for regression
        predictions = self.decoder(pooled_features) 
        return predictions, attn_weights

class MaskedMSELoss(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, predictions, targets):
        valid_mask = ~torch.isnan(targets)
        
        if valid_mask.sum() == 0:
            return torch.tensor(0.0, device=predictions.device, requires_grad=True)
            
        preds_valid = predictions[valid_mask]
        targets_valid = targets[valid_mask]
        
        loss = F.mse_loss(preds_valid, targets_valid)
        return loss

print("Architecture and MaskedMSELoss loaded successfully.")

Initializing Regression-based Architecture and Loss Function...
Architecture and MaskedMSELoss loaded successfully.


In [9]:
import torch
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from IPython.display import display, FileLink

print("============================================================")
print("Initializing Regression Training Engine & Evaluation Pipeline")
print("============================================================\n")

# --- 1. SETUP ---
EPOCHS = 10
EMBED_DIM = 128
SAVE_PATH = "novhyt_regression_best.pth"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SatGenTransformer(embed_dim=EMBED_DIM).to(device)
criterion = MaskedMSELoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

train_losses = []
val_losses = []
best_val_loss = float('inf')

print(f"Starting regression training for {EPOCHS} epochs on {device}...\n")

# --- 2. TRAINING LOOP ---
for epoch in range(EPOCHS):
    model.train()
    epoch_train_loss = 0.0
    train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")
    
    for viirs_batch, lulc_batch, target_batch in train_pbar:
        viirs_batch = viirs_batch.to(device)
        lulc_batch = lulc_batch.to(device)
        target_batch = target_batch.to(device)
        
        optimizer.zero_grad()
        predictions, _ = model(viirs_batch, lulc_batch)
        loss = criterion(predictions, target_batch)
        loss.backward()
        optimizer.step()
        
        epoch_train_loss += loss.item()
        train_pbar.set_postfix({'mse_loss': f"{loss.item():.4f}"})
        
    avg_train_loss = epoch_train_loss / len(train_loader)
    train_losses.append(avg_train_loss)
    
    model.eval()
    epoch_val_loss = 0.0
    with torch.no_grad():
        val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Valid]")
        for viirs_batch, lulc_batch, target_batch in val_pbar:
            viirs_batch = viirs_batch.to(device)
            lulc_batch = lulc_batch.to(device)
            target_batch = target_batch.to(device)
            
            predictions, _ = model(viirs_batch, lulc_batch)
            loss = criterion(predictions, target_batch)
            
            epoch_val_loss += loss.item()
            val_pbar.set_postfix({'mse_loss': f"{loss.item():.4f}"})
            
    avg_val_loss = epoch_val_loss / len(val_loader)
    val_losses.append(avg_val_loss)
    
    # Formal Academic Log Print
    print("\n" + "-"*50)
    print(f"EPOCH {epoch+1} STATISTICAL REPORT")
    print(f"Average Training MSE   : {avg_train_loss:.6f}")
    print(f"Average Validation MSE : {avg_val_loss:.6f}")
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), SAVE_PATH)
        print("Status: Validation MSE improved. Model parameters saved.")
    else:
        print("Status: No improvement in Validation MSE.")
    print("-" * 50 + "\n")

print("Training Complete! Generating Paper Visualizations...\n")

# --- 3. LEARNING CURVE PLOT ---
plt.figure(figsize=(10, 6))
plt.plot(range(1, EPOCHS+1), train_losses, label='Train MSE', marker='o', color='blue')
plt.plot(range(1, EPOCHS+1), val_losses, label='Validation MSE', marker='s', color='orange')
plt.title('Learning Curve: Mean Squared Error (Regression)')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend()
learning_curve_file = 'learning_curve_regression.png'
plt.savefig(learning_curve_file, bbox_inches='tight', dpi=300)
plt.close()

# --- 4. INFERENCE & ERROR MAP ---
print("Running inference on unseen test data...")

model.load_state_dict(torch.load(SAVE_PATH))
model.eval()

viirs_batch, lulc_batch, target_batch = next(iter(test_loader))

with torch.no_grad():
    predictions, _ = model(viirs_batch[0:1].to(device), lulc_batch[0:1].to(device))
    pred_map = predictions.cpu()[0, 0]

target_clean = torch.nan_to_num(target_batch[0, 0], nan=0.0)
error_map = torch.abs(pred_map - target_clean).numpy()

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Spatiotemporal Fire Regression: Continuous Value Prediction', fontsize=18)

for i in range(3):
    axes[0, i].imshow(viirs_batch[0, i, 0].cpu().numpy(), cmap='viridis')
    axes[0, i].set_title(f"Input Day {i+1} (Thermal Sequence)")
    axes[0, i].axis('off')

vmin = min(target_clean.min().item(), pred_map.min().item())
vmax = max(target_clean.max().item(), pred_map.max().item())

im1 = axes[1, 0].imshow(target_clean.numpy(), cmap='inferno', vmin=vmin, vmax=vmax)
axes[1, 0].set_title("Actual Fire Truth (Continuous)")
axes[1, 0].axis('off')
fig.colorbar(im1, ax=axes[1, 0], fraction=0.046, pad=0.04)

im2 = axes[1, 1].imshow(pred_map.numpy(), cmap='inferno', vmin=vmin, vmax=vmax)
axes[1, 1].set_title("Model Prediction")
axes[1, 1].axis('off')
fig.colorbar(im2, ax=axes[1, 1], fraction=0.046, pad=0.04)

im3 = axes[1, 2].imshow(error_map, cmap='hot')
axes[1, 2].set_title("Absolute Error Map (Black=Perfect, Yellow=High Error)")
axes[1, 2].axis('off')
fig.colorbar(im3, ax=axes[1, 2], fraction=0.046, pad=0.04)

plt.tight_layout()
eval_maps_file = 'regression_evaluation_maps.png'
plt.savefig(eval_maps_file, bbox_inches='tight', dpi=300)
plt.close()

# --- 5. GENERATE DOWNLOAD LINKS ---
print("============================================================")
print("DOWNLOAD READY - Click the links below to save to your PC:")
print("============================================================")
display(FileLink(SAVE_PATH))
display(FileLink(learning_curve_file))
display(FileLink(eval_maps_file))

Initializing Regression Training Engine & Evaluation Pipeline

Starting regression training for 10 epochs on cuda...



Epoch 1/10 [Valid]: 100%|██████████| 27/27 [00:54<00:00,  2.02s/it, mse_loss=1.2065]    



--------------------------------------------------
EPOCH 1 STATISTICAL REPORT
Average Training MSE   : 4411.508955
Average Validation MSE : 8140.132058
Status: Validation MSE improved. Model parameters saved.
--------------------------------------------------



Epoch 2/10 [Valid]: 100%|██████████| 27/27 [00:43<00:00,  1.60s/it, mse_loss=2.5535]    



--------------------------------------------------
EPOCH 2 STATISTICAL REPORT
Average Training MSE   : 4385.998027
Average Validation MSE : 8119.819681
Status: Validation MSE improved. Model parameters saved.
--------------------------------------------------



...

Epoch 10/10 [Valid]: 100%|██████████| 27/27 [00:43<00:00,  1.59s/it, mse_loss=1486.4381] 



--------------------------------------------------
EPOCH 10 STATISTICAL REPORT
Average Training MSE   : 3870.601145
Average Validation MSE : 7693.766834
Status: No improvement in Validation MSE.
--------------------------------------------------

Training Complete! Generating Paper Visualizations...

Running inference on unseen test data...
DOWNLOAD READY - Click the links below to save to your PC:


/kaggle/working/novhyt_regression_best.pth

/kaggle/working/learning_curve_regression.png

/kaggle/working/regression_evaluation_maps.png